# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [47]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [48]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (16992, 22)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [49]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5162,17445914.41,3379.68,1167.38,26602
Audio,3949,9234318.00,2338.39,1073.00,15078
Mobilier,3936,8363145.58,2124.78,717.48,18274
Accessoire,3870,7575740.68,1957.56,834.65,14978


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [50]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie   montant  ca_categorie  part_dans_categorie
0  Informatique   457.881  1.744591e+07               0.0026
1      Mobilier   140.960  8.363146e+06               0.0017
2         Audio  2073.360  9.234318e+06               0.0225
3  Informatique  2281.000  1.744591e+07               0.0131
4      Mobilier  1904.490  8.363146e+06               0.0228

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [51]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,5738,P0027,0.165
telephone,2865,P0015,0.166
web,8389,P0038,0.170


### Partie autonome

In [52]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
#     avec les totaux en marge. Utilisez pivot_table avec margins=True.

ca_categorie_canal = sales.pivot_table(
    index='categorie',
    columns='canal',
    values='montant',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
).round(0)

In [53]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.

customer_profile = sales.groupby('id_client').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    nb_categories=('categorie', 'nunique'),
).round(2)

# Dix meilleurs clients par CA
top_10_customers = customer_profile.nlargest(10, 'ca_total')
top_10_customers

,nb_commandes,ca_total,panier_moyen,premiere_commande,derniere_commande,nb_categories
id_client,,,,,,
C00696,7,1110180.19,158597.17,2023-03-13 00:00:00,2024-11-19 00:00:00,3
C02327,4,907834.48,226958.62,2023-10-31 00:00:00,2024-12-28 14:36:00,2
C00279,10,876999.45,87699.95,2023-01-24 00:00:00,2024-11-05 00:00:00,4
C02631,10,788266.88,78826.69,2023-01-12 00:00:00,2024-12-19 00:00:00,4
C00941,7,783179.31,111882.76,2023-01-29 00:00:00,2024-09-15 00:00:00,3
C00355,13,779167.47,59935.96,2023-04-09 16:29:00,2024-09-25 00:00:00,4
C02101,6,750743.90,125123.98,2023-01-05 11:00:00,2024-02-03 00:00:00,3
C00806,5,707386.06,141477.21,2023-05-03 00:00:00,2024-08-26 00:00:00,2
C02285,7,687164.51,98166.36,2023-01-10 10:43:00,2024-12-27 00:00:00,2


In [54]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?

sales['mediane_categorie'] = sales.groupby('categorie')['montant'].transform('median')
sales['est_montant_eleve'] = sales['montant'] > 1.5 * sales['mediane_categorie']

part_elevees = sales['est_montant_eleve'].mean()
print(f"{sales['est_montant_eleve'].sum()} commandes concernées ({part_elevees:.2%})")

5662 commandes concernées (33.32%)


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [55]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 16992
Après jointure produits : 16992
Lignes sans produit correspondant : 75 (0.44%)
Identifiants fautifs : <ArrowStringArray>
['P9999']
Length: 1, dtype: str


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [56]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
16992 lignes -> 17287 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [57]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
try:
    sales.merge(customers_with_duplicates, on='id_client', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, ':', error)

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 id_client
   C00001
   C00002
   C00003
   C00004
   C00005 ...


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [58]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).

row_counts = {'sales (départ)': len(sales)}


def join_step(left, right, key, validate, name, suffix):
    """Jointure left avec validate, indicateur d'orphelins et comptage des lignes."""
    merged = left.merge(right, on=key, how='left', validate=validate,
                        suffixes=('', suffix), indicator=f'_merge_{name}')
    merged[f'orphelin_{name}'] = merged[f'_merge_{name}'].eq('left_only')
    merged = merged.drop(columns=f'_merge_{name}')

    n_orphans = merged[f'orphelin_{name}'].sum()
    row_counts[f'après {name}'] = len(merged)
    print(f'{name:<10} {len(left)} -> {len(merged)} lignes | '
          f'{n_orphans} orphelines ({n_orphans / len(merged):.2%})')
    assert len(merged) == len(left), f'Explosion ou perte de lignes sur {name}'
    return merged


analytic = sales.copy()
analytic = join_step(analytic, products,  'id_produit', 'many_to_one', 'produit', '_prod')
analytic = join_step(analytic, customers, 'id_client',  'many_to_one', 'client',  '_client')
analytic = join_step(analytic, stores,    'id_magasin', 'many_to_one', 'magasin', '_magasin')

print()
print(pd.Series(row_counts, name='nb_lignes'))
print('Colonnes :', analytic.shape[1])

produit    16992 -> 16992 lignes | 75 orphelines (0.44%)
client     16992 -> 16992 lignes | 0 orphelines (0.00%)
magasin    16992 -> 16992 lignes | 0 orphelines (0.00%)

sales (départ)    16992
après produit     16992
après client      16992
après magasin     16992
Name: nb_lignes, dtype: int64
Colonnes : 41


In [59]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.

def safe_merge(left, right, **kwargs):
    """Jointure pandas qui échoue bruyamment si un left join change le nombre de lignes.

    Accepte les mêmes arguments que DataFrame.merge (on, how, validate, suffixes...).
    Affiche le taux de lignes de `left` sans correspondance dans `right`.
    """
    if 'indicator' in kwargs:
        raise ValueError("safe_merge gère déjà 'indicator' : ne le passez pas.")

    how = kwargs.get('how', 'inner')  # 'inner' est la valeur par défaut de pandas
    merged = left.merge(right, indicator='_merge_check', **kwargs)

    n_unmatched = (merged['_merge_check'] == 'left_only').sum()
    unmatched_rate = n_unmatched / len(merged) if len(merged) else 0.0
    merged = merged.drop(columns='_merge_check')

    print(f'safe_merge ({how}) : {len(left)} -> {len(merged)} lignes | '
          f'{n_unmatched} clés non appariées ({unmatched_rate:.2%})')

    if how == 'left' and len(merged) != len(left):
        raise ValueError(
            f"Le nombre de lignes a changé lors d'un left join : "
            f'{len(left)} -> {len(merged)}. La clé de droite n\'est probablement pas unique.'
        )
    return merged

---
## Atelier 6.3 — Formats long et large (35 min)

In [60]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{month:02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,367032.0,296781.0,333273.0,1218847.0,539612.0,339807.0,334524.0,1204402.0,491953.0,1090514.0,893844.0,465151.0
Audio,1180732.0,658118.0,389644.0,1082484.0,427252.0,1169529.0,575026.0,1057970.0,418011.0,431277.0,1169598.0,674677.0
Informatique,1808515.0,1586185.0,2721960.0,676080.0,2323577.0,750591.0,2923108.0,1046418.0,565554.0,596944.0,773439.0,1673544.0
Mobilier,849867.0,328436.0,369598.0,1133552.0,1356966.0,873502.0,361750.0,195968.0,1405982.0,300714.0,715121.0,471688.0


In [61]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,367032.0
1,Audio,mois_01,1180732.0
2,Informatique,mois_01,1808515.0
3,Mobilier,mois_01,849867.0
4,Accessoire,mois_02,296781.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [66]:
# Q6. Repasser du format long au format large sans utiliser pivot_table,
#     puis vérifier que vous retrouvez exactement le tableau de départ.
back_to_wide = (
    long.pivot(index='categorie', columns='mois', values='ca')
        .rename_axis(columns=None)   # supprime le nom d'axe 'mois' ajouté par pivot
)

# Vérification : exactement le tableau de départ (valeurs, index, colonnes, dtypes)
pd.testing.assert_frame_equal(wide, back_to_wide)
print('OK : identique au tableau de départ')

OK : identique au tableau de départ


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [68]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [72]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.
# Q7. Modèle en étoile : table de faits + 4 dimensions, exportés en Parquet
STAR_DIR = PROCESSED_DIR / 'star'
STAR_DIR.mkdir(parents=True, exist_ok=True)

# --- Table de faits : clés étrangères + mesures + statut ---
fact_sales = (
    sales.assign(
        cle_date=pd.to_numeric(sales['date_commande'].dt.strftime('%Y%m%d')).astype('Int64')
    )
    .rename(columns={'montant_apres_remise': 'montant_net'})
    [['cle_date', 'id_client', 'id_produit', 'id_magasin', 'quantite',
      'prix_unitaire', 'remise_pct', 'montant', 'montant_net', 'statut']]
)
print('Dates manquantes dans les faits :', fact_sales['cle_date'].isna().sum())

# --- Dimensions : une ligne par clé primaire ---
dim_customer = customers.drop_duplicates(subset='id_client').reset_index(drop=True)
dim_product = products.drop_duplicates(subset='id_produit').reset_index(drop=True)
dim_store = stores.drop_duplicates(subset='id_magasin').reset_index(drop=True)
# dim_date est déjà construite plus haut


# --- Clés orphelines : membre « Inconnu » ajouté à la dimension ---
def add_unknown_members(dim, key, fact_keys, label='Inconnu'):
    """Ajoute une ligne à la dimension pour chaque clé présente dans les faits mais absente de la dimension."""
    missing = pd.Index(fact_keys.dropna().unique()).difference(dim[key])
    if len(missing) == 0:
        return dim
    unknown = pd.DataFrame({key: missing})
    for col in dim.columns.drop(key):
        unknown[col] = label if pd.api.types.is_string_dtype(dim[col]) else np.nan
    print(f'{key} : {len(missing)} clé(s) ajoutée(s) comme « {label} » -> {list(missing)}')
    return pd.concat([dim, unknown], ignore_index=True)


dim_customer = add_unknown_members(dim_customer, 'id_client', fact_sales['id_client'])
dim_product = add_unknown_members(dim_product, 'id_produit', fact_sales['id_produit'])
dim_store = add_unknown_members(dim_store, 'id_magasin', fact_sales['id_magasin'])

# --- Intégrité référentielle ---
dimensions = {
    'cle_date': dim_date,
    'id_client': dim_customer,
    'id_produit': dim_product,
    'id_magasin': dim_store,
}
for key, dim in dimensions.items():
    assert dim[key].is_unique, f'Clé non unique dans la dimension {key}'
    n_orphans = (~fact_sales[key].isin(dim[key])).sum()
    print(f'{key:<12} clés orphelines : {n_orphans}')

# --- Export Parquet ---
tables = {'fact_sales': fact_sales, 'dim_date': dim_date, 'dim_customer': dim_customer,
          'dim_product': dim_product, 'dim_store': dim_store}
for name, table in tables.items():
    table.to_parquet(STAR_DIR / f'{name}.parquet', index=False)
    print('écrit :', f'{name}.parquet', table.shape)

Dates manquantes dans les faits : 0
id_produit : 1 clé(s) ajoutée(s) comme « Inconnu » -> ['P9999']
cle_date     clés orphelines : 0
id_client    clés orphelines : 0
id_produit   clés orphelines : 0
id_magasin   clés orphelines : 0
écrit : fact_sales.parquet (16992, 10)
écrit : dim_date.parquet (730, 10)
écrit : dim_customer.parquet (3060, 5)
écrit : dim_product.parquet (40, 5)
écrit : dim_store.parquet (9, 5)


In [73]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.

def check_referential_integrity(fact_table, dimensions):
    """dimensions : dict {nom_colonne_cle: dataframe_dimension}

    Affiche, pour chaque clé, le nombre de valeurs de la table de faits absentes
    de la dimension, puis lève une exception si au moins une clé est orpheline.
    """
    failures = {}

    for key, dim in dimensions.items():
        if key not in fact_table.columns:
            raise KeyError(f"Clé '{key}' absente de la table de faits")
        if key not in dim.columns:
            raise KeyError(f"Clé '{key}' absente de la dimension")
        if not dim[key].is_unique:
            raise ValueError(f"La clé '{key}' n'est pas unique dans la dimension")

        is_orphan = ~fact_table[key].isin(dim[key])
        n_orphans = int(is_orphan.sum())
        print(f'{key:<12} {n_orphans:>5} clé(s) orpheline(s) '
              f'({n_orphans / len(fact_table):.2%} des lignes)')

        if n_orphans:
            failures[key] = fact_table.loc[is_orphan, key].unique()[:5].tolist()

    if failures:
        details = '; '.join(f'{key} : {values}' for key, values in failures.items())
        raise AssertionError(f'Intégrité référentielle violée -> exemples : {details}')

    print('OK : toutes les clés de la table de faits existent dans leur dimension.')

---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.